In [ ]:
import requests
import time
import json
import base64
import msgpack
import threading
import websocket
import ast
import random
from msgpack import ExtType


# --- Helpers ---

def encode(data) -> str:
    packed = msgpack.packb(data, use_bin_type=True)
    return "b" + base64.b64encode(packed).decode("utf-8")

def decode(text):
    try:
        return msgpack.unpackb(base64.b64decode(text[1:]), raw=False)
    except:
        try:
            return msgpack.unpackb(base64.b64decode(text), raw=False)
        except:
            return text

def params(sid=None):
    p = {"EIO": 4, "transport": "polling", "t": str(int(time.time() * 1000))}
    if sid:
        p["sid"] = sid
    return p

def send_action(ws, action, args):
    payload = encode({
        "type": 2,
        "data": ["message", {"action": action, "args": args}],
        "options": {"compress": True},
        "nsp": "/"
    })
    ws.send(payload)

with open("players.json", 'r') as file:
    players = json.load(file)
    
def player_dump():
    with open("players.json",'w') as file:
        json.dump(players, file)
        

#need real cave, boiler room, tourhq, lighthouse, top of lighthouse, some secret stuff idk
rooms = {"attic": 221, "beach": 400, "iceberg":  805, "book": 111, "cave": 813, "coffee": 110, "cove": 810, "club": 120, "dock": 800, "dojo": 320, "dojo_yard": 321, "forest": 809, "forts": 801,
         "lodge": 220, "lounge": 121, "hill": 230, "pet": 310, "plaza": 300, "rink": 802, "mine_shack": 808, "shop": 130, "sport": 210, "town": 300, "village": 200, "theatre": 340, "plant": 122, "welcome_room": 1 }



# --- Login + Join + WebSocket ---

def main():
    session = requests.Session()
    url = "https://play.cpjourney.net/world/login/"
    url2 = "https://play.cpjourney.net/world/marshmallow/"
    headers = {"Content-Type": "text/plain;charset=UTF-8"}

    # Login
    sid = json.loads(session.get(url, params=params()).text[1:])["sid"]
    session.post(url, params=params(sid), data=encode({"type": 0, "data": ExtType(code=0, data=b'\x00'), "nsp": "/"}), headers=headers)
    time.sleep(0.5)
    session.get(url, params=params(sid))
    session.post(url, params=params(sid), data=encode({
        "type": 2,
        "data": ["message", {"action": "login", "args": {
            "username": "",
            "password": "",
            "secret": ""
        }}],
        "options": {"compress": True},
        "nsp": "/"
    }), headers=headers)
    time.sleep(0.5)
    login_reply = decode(session.get(url, params=params(sid)).text)
    key = login_reply["data"][1]["args"]["key"]
    print("Got key:", key)

    # World server
    server_sid = json.loads(session.get(url2, params=params()).text[1:])["sid"]
    session.post(url2, params=params(server_sid), data=encode({"type": 0, "data": ExtType(code=0, data=b'\x00'), "nsp": "/"}), headers=headers)
    time.sleep(0.5)
    session.get(url2, params=params(server_sid))
    session.post(url2, params=params(server_sid), data=encode({ 
        "type": 2,
        "data": ["message", {"action": "game_auth", "args": {
            "username": "",
            "key": key,
            "createToken": False,
            "joinInvis": False,
            "takeoverMascot": ExtType(code=0, data=b'\x00')
        }}],
        "options": {"compress": True},
        "nsp": "/"
    }), headers=headers)
    time.sleep(0.5)
    session.get(url2, params=params(server_sid))
    session.post(url2, params=params(server_sid), data=encode({
        "type": 2,
        "data": ["message", {"action": "join_server", "args": {}}],
        "options": {"compress": True},
        "nsp": "/"
    }), headers=headers)
    print("Joined! Upgrading to WebSocket...")

    # --- WebSocket handlers ---
    def on_game_message(ws, message):
        if message == "2":
            ws.send("3")
            return
        try:
            if isinstance(message, bytes):
                decoded = msgpack.unpackb(message, raw=False)
            else:
                decoded = msgpack.unpackb(base64.b64decode(message), raw=False)
                
            data = decoded.get("data", [])

            if len(data) > 1 and isinstance(data[1], dict):
                action = data[1].get('action',{})
                args = data[1].get("args", {})

                with open("actions.txt", "a") as f:
                    f.write(f"{data}\n")
                
                if data[1]['action'] == 'send_message':
                    with open('chat_data.txt',"a") as f:
                        f.write(f"{args.get('message',{})}\n")
                #['message', {'action': 'add_player', 'args': {'user': {'id': 274747, 'username': 'Santa CIause', 'displayName': 'Santa CIause', 'joinTime': ExtType(code=0, data=b'\x00\x00\x01\x93\xf0)>\x18'), 'hat': 0, 'head': 414, 'face': 3037, 'face_mask': 0, 'neck': 0, 'neck_scarf': 0, 'body': 4126, 'body_shirt': 0, 'hand': 5195, 'hand_glove': 0, 'feet': 0, 'color': 5, 'photo': 0, 'flag': 0, 'transform': 0, 'x': 786, 'y': 676, 'frame': 1, 'walking': 0, 'walkingPuffleType': ExtType(code=0, data=b'\x00'), 'openSprite': ExtType(code=0, data=b'\x00'), 'mascotGiveaway': ExtType(code=0, data=b'\x00'), 'iglooOpen': 0, 'iglooBounds': 0, 'igloo_slot': 0, 'currentLayer': 1, 'fireRank': 0}}}, 'Pv6NPRj.0']

                if data[1]['action'] == "add_player":
                    if str(data[1]['args']['user']['id']) not in players:
                        players[str(data[1]['args']['user']['id'])] = {"username": data[1]['args']['user']['username'],
                                                                  "displayName": data[1]['args']['user']['displayName']}

        except:
            if message not in [b"", "3", "5"]:
                #print("Undecodable:", message)
                pass
                
        

    def on_message_with_probe(ws, message):
        if message == "3probe":
            ws.send("5")
            print("Upgraded! Listening for game events...")
            ws.on_message = on_game_message
        elif message == "2":
            ws.send("3")
        else:
            on_game_message(ws, message)

    def on_open(ws):
        print("WebSocket connected! Sending probe...")
        ws.send("2probe")

    def on_error(ws, error):
        print("WS Error:", error)

    def on_close(ws, close_status, close_msg):
        print("WS Closed:", close_status, close_msg)

    ws = websocket.WebSocketApp(
        f"wss://play.cpjourney.net/world/marshmallow/?EIO=4&transport=websocket&sid={server_sid}",
        on_open=on_open,
        on_message=on_message_with_probe,
        on_error=on_error,
        on_close=on_close,
        header={"Origin": "https://play.cpjourney.net", "User-Agent": "Mozilla/5.0"}
    )

    ws_thread = threading.Thread(target=ws.run_forever, daemon=True)
    ws_thread.start()

    return ws


# --- Entry point ---

def follow(id):
    with open('actions.txt',"r") as file:
        #end of file
        file.seek(0,2)
        
        while True:
            line = file.readline()

            if not line:
                time.sleep(0.1)
                continue
            line = line.strip()
            try:
                data = ast.literal_eval(line)
            except ValueError as e:
                print(e)
                print(line.strip)
            if data[1]['action'] == 'send_position' and int(data[1]['args']['id']) == id:
                send_action(ws, "send_position", {"x": int(data[1]['args']['x'] + random.randint(1,50)), "y": int(data[1]['args']['y'])+ random.randint(1,50)})

def greet():
    name = "Flint"
    greetings = ['Hiya', 'Whats up', 'Nice to see you', 'leave me alone bro', 'Hey', 'Sup']
    with open('actions.txt', "r") as file:
        file.seek(0,2)
    #['message', {'action': 'send_message', 'args': {'id': 274747, 'message': 'YOOO Flint!!'}}, 'PuicmgV.
        while True:
            line = file.readline()

            if not line:
                time.sleep(0.1)
                continue
            line.strip()

            try:
                data = ast.literal_eval(line)
            except ValueError as e:
                pass
            try:
                if data[1]['action'] == 'send_message' and name.lower() in data[1]['args']['message'].lower():
                    send_action(ws, 'send_message', {"message": greetings[random.randint(0,(len(greetings)) - 1 )]})
                    send_action(ws, 'send_frame', {'set': False, 'frame': 25})
            except UnboundLocalError as e:
                pass

def mine():
    send_action(ws,'send_frame', {'set': True, 'frame': 26})
    send_action(ws,'mine',{})
    last_mine_time = time.time()

    with open('actions.txt',"r") as file:
        file.seek(0,2)
        while True:
            line = file.readline()

            if not line:
                time.sleep(.1)
                continue
            line.strip()
            try:
                data = ast.literal_eval(line)
            except ValueError as e:
                pass
            #['message', {'action': 'send_message', 'args': {'id': 274747, 'message': 'flint'}}, 'PvHXV5y.8']
            try:
                if data[1]['action'] == 'send_message' and data[1]['args']['id'] == 274747 and data[1]['args']['message'] == 'shifts up':
                    time.sleep(2)
                    send_action(ws,"send_message", {"message": "IM FREE!!"})
                    time.sleep(1)
                    send_action(ws,"join_room",{'room':rooms['welcome_room']})
                    return False
            except UnboundLocalError as e:
                pass

            if time.time() - last_mine_time >= 60:
                    send_action(ws,'send_frame', {'set': True, 'frame': 26})
                    send_action(ws,'mine',{})
                    last_mine_time = time.time()
            


def bag_fries():

    body = ["hand","neck","body","head","feet", "face"]
    send_action(ws,"join_room",{'room':rooms['cave']})
    time.sleep(.5)
    send_action(ws, "send_message",{"message": "Time to put the fries in the bag!"})
    time.sleep(.5)
    for values in body:
        send_action(ws,"remove_item", {'type': values})
        time.sleep(.2)
    
    send_action(ws,'update_player', {'item': 429})
    time.sleep(.5)
    send_action(ws,"send_position", {'x': 825 + random.randint(-15,15), 'y': 734 + random.randint(-15,15)})

def make_money():
    start = "get to work"
    working = False
    

    with open('actions.txt',"r") as file:
        file.seek(0,2)
        while True:
            line = file.readline()
            if not line:
                time.sleep(.1)
                continue
            try:
                data = ast.literal_eval(line)
            except ValueError as e:
                pass
#['message', {'action': 'send_message', 'args': {'id': 274747, 'message': 'flint'}}, 'PvHXV5y.8']
            try: 
                if not working and data[1]['action'] == 'send_message' and data[1]['args']['id'] == 274747 and start.lower() in data[1]['args']['message'].lower():
                    working = True
                    time.sleep(2)
                    bag_fries()
                    working  = mine()
            except (UnboundLocalError, KeyError) as e:
                pass
            



if __name__ == "__main__":
    
    ws = main()

    time.sleep(1)  # wait for WS to fully connect
# {'action': 'join_room', 'args': {'room': 330, 'x': 789, 'y': 468}}
    # Do stuff here
    #send_action(ws, "send_message", {"message": "Hello!"})
    #send_action(ws, "send_position", {"x": 1010, "y": 800})
    send_action(ws, "send_message",  {"message": "Hello!"})


    
    #send_action(ws, "join_room", {'room': 300})
    threading.Thread(target = make_money, daemon = True).start()
    threading.Thread(target = greet, daemon = True).start()
    

    #follow(274747)
    
    

    # Keep alive
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        player_dump()
        print("Stopped.")
        ws.close()

Got key: f7b3de02c407ad31089840c04e60609c112d34db8b1c3b8fc1f4eb030be01226
Joined! Upgrading to WebSocket...
WebSocket connected! Sending probe...
Upgraded! Listening for game events...


In [ ]:
import requests
import time
import json
import base64
import msgpack
import threading
import websocket
import ast
import random
from msgpack import ExtType


# --- Helpers ---

def encode(data) -> str:
    packed = msgpack.packb(data, use_bin_type=True)
    return "b" + base64.b64encode(packed).decode("utf-8")

def decode(text):
    try:
        return msgpack.unpackb(base64.b64decode(text[1:]), raw=False)
    except:
        try:
            return msgpack.unpackb(base64.b64decode(text), raw=False)
        except:
            return text

def params(sid=None):
    p = {"EIO": 4, "transport": "polling", "t": str(int(time.time() * 1000))}
    if sid:
        p["sid"] = sid
    return p

def send_action(ws, action, args):
    payload = encode({
        "type": 2,
        "data": ["message", {"action": action, "args": args}],
        "options": {"compress": True},
        "nsp": "/"
    })
    ws.send(payload)


In [ ]:
#wave
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26qc2VuZF9mcmFtZaRhcmdzgqNzZXTCpWZyYW1lGadvcHRpb25zgahjb21wcmVzc8OjbnNwoS8="))

In [ ]:

#need real cave, boiler room, tourhq, lighthouse, top of lighthouse, some secret stuff idk
rooms = {"attic": 221, "beach": 400, "iceberg":  805, "book": 111, "cave": 813, "coffee": 110, "cove": 810, "club": 120, "dock": 800, "dojo": 320, "dojo_yard": 321, "forest": 809, "forts": 801,
         "lodge": 220, "lounge": 121, "hill": 230, "pet": 310, "plaza": 300, "rink": 802, "mine_shack": 808, "shop": 130, "sport": 210, "town": 300, "village": 200, "theatre": 340, "plant": 122, "welcome_room": 1 }

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26tc2VuZF9wb3NpdGlvbqRhcmdzgqF4zQMpoXnNAuKnb3B0aW9uc4GoY29tcHJlc3PDo25zcKEv"))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26tdXBkYXRlX3BsYXllcqRhcmdzgaRpdGVtzQGtp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRoZWFkp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26qc2VuZF9mcmFtZaRhcmdzgqNzZXTDpWZyYW1lGqdvcHRpb25zgahjb21wcmVzc8OjbnNwoS8="))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRoZWFkp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRuZWNrp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("g6R0eXBlAqRkYXRhk6dtZXNzYWdlgqZhY3Rpb26tdXBkYXRlX3BsYXllcqRhcmdzg6JpZM4AB8FnpGl0ZW0ApHNsb3SpZmFjZV9tYXNrp1B2SFZsUGOjbnNwoS8="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRib2R5p29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRib2R5p29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRoYW5kp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRmZWV0p29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26rcmVtb3ZlX2l0ZW2kYXJnc4GkdHlwZaRmYWNlp29wdGlvbnOBqGNvbXByZXNzw6Nuc3ChLw=="))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26tc2VuZF9wb3NpdGlvbqRhcmdzgqF4zQM5oXnNAt6nb3B0aW9uc4GoY29tcHJlc3PDo25zcKEv"))

In [ ]:
print(random.randint(-10,10))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26qc2VuZF9mcmFtZaRhcmdzgqNzZXTDpWZyYW1lGqdvcHRpb25zgahjb21wcmVzc8OjbnNwoS8="))

In [ ]:
print(decode("hKR0eXBlAqRkYXRhkqdtZXNzYWdlgqZhY3Rpb26kbWluZaRhcmdzgKdvcHRpb25zgahjb21wcmVzc8OjbnNwoS8="))

In [ ]:
start = "get to work"
test= "flint its time to get to work"

print(start.lower() in test.lower())